![LogoUC3M](https://upload.wikimedia.org/wikipedia/commons/thumb/a/a6/Acr%C3%B3nimo_y_nombre_de_la_UC3M.svg/320px-Acr%C3%B3nimo_y_nombre_de_la_UC3M.svg.png)

# 2DA PRACTICA - DETERMINACIÓN DE TIPOS DE ESTRELLAS

### **Grupo 82 - Equipo 15**

*   Ariana Cornejo Infante,     100522121, 100522121@alumnos.uc3m.es
*   Francisco Pérez Sokolowski, 100522254, 100522254@alumnos.uc3m.es

El objetivo es aplicar técnicas de **aprendizaje no supervisado** para detectar agrupaciones naturales de estrellas. Se cuenta con la información física de 240 estrellas en el fichero `stars_data.csv` que contiene las siguientes variables:

- Temperature → temperatura superficial (Kelvin)
- L → luminosidad relativa al Sol
- R → radio relativo al Sol
- A_M → magnitud absoluta
- Color → color dominante del espectro
- Spectral_Class → clase espectral



In [ ]:
# Importaciones necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

# Fijar semilla (usando el NIA) para reproducibilidad
SEED = 100522121
np.random.seed(SEED)

## 1. Análisis exploratorio inicial

Antes de aplicar cualquier técnica de clustering:

1. Revisamos estructura del dataset
2. Detectamos valores nulos
3. Observamos estadísticas básicas

In [ ]:
# Carga del dataset del fichero starts_data.csv
df = pd.read_csv("https://github.com/100522254/Proyecto-ML-2/raw/main/stars_data.csv")
df.head()

df.info()
df.describe()
df.isnull().sum()

### 1.1. Codificación ordinal de variables categóricas

Las variables categóricas **Color** y **Spectral_Class** deben codificarse ordinalmente porque contienen información física relacionada con la temperatura y energía de la estrella. Asignamos valores numéricos manteniendo el orden físico según la clase espectral: ***O > B > A > F > G > K > M***

In [ ]:
# Codificación ordinal
spectral_order = {
    "O": 6,
    "B": 5,
    "A": 4,
    "F": 3,
    "G": 2,
    "K": 1,
    "M": 0
}

df["Spectral_Class"] = df["Spectral_Class"].map(spectral_order)

# Codificación del color. Orden físico típico:
# Red → Orange → Yellow → White → Blue-white → Blue
color_order = {
    "Red": 0,
    "Orange": 1,
    "Yellow": 2,
    "Yellow-White": 3,
    "White": 4,
    "Blue-White": 5,
    "Blue": 6
}

df["Color"] = df["Color"].map(color_order)

### 1.2. Normalización de variables

Antes de aplicar PCA es necesario escalar las variables porque tienen diferentes unidades físicas:

- temperatura (Kelvin)
- radio solar relativo
- luminosidad relativa
- magnitud absoluta

Se utiliza **StandardScaler**, que transforma los datos con media 0 y desviación estándar 1.

In [ ]:
# Normalizar variables con StandardScaler
scaler = StandardScaler()

X_scaled = scaler.fit_transform(df)

## 2. Reducción de dimensionalidad mediante PCA

Para facilitar la visualización y mejorar el clustering, reducimos la dimensionalidad a 2 componentes principales. PCA permite:

- eliminar redundancia
- conservar máxima varianza
- facilitar representación visual

In [ ]:
# PCA
pca = PCA(n_components=2, random_state=SEED)

X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada:", pca.explained_variance_ratio_)

# VISUALIZACIÓN PCA
plt.figure(figsize=(8,6))

plt.scatter(X_pca[:,0], X_pca[:,1])

plt.title("Representación PCA de las estrellas")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")

plt.show()

## 3. Aplicación de tres algoritmos de clustering
### 3.1. Aplicación del algoritmo K-Means

*   **Selección del número óptimo de clusters (Método del codo)**

Permite estimar el número óptimo de clusters observando la inercia dentro del cluster.
Aplicamos K-Means usando el número de clusters obtenido con el método del codo.

In [ ]:
# MÉTODO DEL CODO
inertia = []

for k in range(1,10):
    kmeans = KMeans(n_clusters=k, random_state=SEED)
    kmeans.fit(X_pca)
    inertia.append(kmeans.inertia_)

plt.plot(range(1,10), inertia, marker="o")

plt.title("Método del codo")
plt.xlabel("Número de clusters")
plt.ylabel("Inercia")

plt.show()

# Aplicar kmeans
kmeans = KMeans(n_clusters=3, random_state=SEED)

labels_kmeans = kmeans.fit_predict(X_pca)

# VISUALIZACIÓN CLUSTERS
plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=labels_kmeans,
    cmap="viridis"
)

plt.title("Clusters obtenidos con KMeans")

plt.show()

*   **Evaluación del clustering mediante Silhouette Score**

El índice silhouette mide la cohesión interna y la separación entre clusters.

Valores cercanos a 1 indican clustering adecuado.

In [ ]:
silhouette = silhouette_score(X_pca, labels_kmeans)

print("Silhouette Score:", silhouette)

### 3.2. Clustering jerárquico aglomerativo

Aplicamos clustering jerárquico como alternativa a KMeans para comparar resultados. Segundo algoritmo

In [ ]:
# CLUSTERING JERÁRQUICO

agglo = AgglomerativeClustering(n_clusters=3)

labels_agglo = agglo.fit_predict(X_pca)

# VISUALIZACIÓN JERÁRQUICA
plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=labels_agglo,
    cmap="plasma"
)

plt.title("Clusters jerárquicos")

plt.show()